# Protein Pretrain (Unified)

This notebook replaces the separate from-scratch, resume, and streaming notebooks. It loads `config/train.64gb.2gpu.yaml` by default for 64GB RAM + 2 GPU training, or a custom YAML through `MDNAC_TRAIN_CONFIG`.

Supports:
- **Windows local/Jupyter**: notebook bootstrap forces safe worker settings when needed.
- **Ubuntu/Linux/cloud VM**: same config, CUDA auto-detection, streaming train parts.
- **Google Colab**: set `MDNAC_REPO_DIR` or clone/mount the repo under `/content` or Google Drive.
- **train_from_scratch**, **resume**, and **auto** modes.

Default config: `config/train.64gb.2gpu.yaml`. Use `MDNAC_TRAIN_CONFIG` only when you want to point at another YAML. Use `MDNAC_REPO_DIR` only when the notebook is not opened from inside the cloned repo.

Data source:
- Local `train.txt` or `train_part_*.txt` files only

After training, the model checkpoint can be uploaded to **HuggingFace** if `HF_TOKEN` is set.

> Put `train_part_*.txt` files in `data/cache/protein_train_parts/`, or provide `data/compiled/refseq_bacteria_protein/train.txt`.


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/MDNAC"),
        Path("/content/drive/MyDrive/MDNAC"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import bootstrap_notebook, materialize_notebook_training_config
from libs.core.pretrain.training_config import load_protein_training_config
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(f"Setup complete: {REPO_DIR}")
RUNTIME


In [ ]:
DEFAULT_TRAIN_CONFIG = REPO_DIR / "config" / "train.64gb.2gpu.yaml"


def activate_training_config(path):
    global CONFIG_SOURCE_PATH, CONFIG_PATH, CONFIG_NOTEBOOK_OVERRIDES, config
    source_path = Path(path).expanduser()
    if not source_path.is_absolute():
        source_path = (REPO_DIR / source_path).resolve()
    CONFIG_SOURCE_PATH = source_path
    CONFIG_PATH, CONFIG_NOTEBOOK_OVERRIDES = materialize_notebook_training_config(
        CONFIG_SOURCE_PATH,
        project_root=REPO_DIR,
    )
    config = load_protein_training_config(REPO_DIR, config_path=CONFIG_PATH)
    return config


CONFIG_SOURCE_PATH = Path(os.environ.get("MDNAC_TRAIN_CONFIG", DEFAULT_TRAIN_CONFIG)).expanduser()
config = activate_training_config(CONFIG_SOURCE_PATH)
local_train_part_dir = config["paths"]["train_part_cache_dir"]
local_train_parts = sorted(local_train_part_dir.glob(config["data"]["train_part_glob"]))
train_text_path = config["paths"]["train_text_path"]
if not local_train_parts and not train_text_path.exists():
    raise FileNotFoundError(
        "Local training data not found. Expected train_part_*.txt in "
        f"{local_train_part_dir} or train.txt at {train_text_path}."
    )

grad_accum = config["training"].get("gradient_accumulation_steps", 1)
config_summary = {
    "source_config_path": str(CONFIG_SOURCE_PATH),
    "effective_config_path": str(CONFIG_PATH),
    "notebook_overrides": CONFIG_NOTEBOOK_OVERRIDES,
    "mode": config["mode"],
    "optimizer_type": config["optimizer"]["type"],
    "device": config["runtime"]["device"],
    "multi_gpu_mode": config["runtime"]["multi_gpu_mode"],
    "data_parallel_device_ids": config["runtime"].get("data_parallel_device_ids"),
    "mixed_precision": config["runtime"]["mixed_precision"],
    "checkpoint_dir": str(config["paths"]["checkpoint_dir"]),
    "resume_state_path": str(config["paths"]["resume_state_path"]),
    "train_part_cache_dir": str(config["paths"]["train_part_cache_dir"]),
    "train_part_glob": config["data"]["train_part_glob"],
    "train_text_path": str(train_text_path),
    "local_train_parts": len(local_train_parts),
    "minio_enabled": bool(config["minio"]["train_parts_prefix_uri"] or config["minio"]["train_part_uris"]),
    "num_epochs": config["training"]["num_epochs"],
    "max_steps": config["training"].get("max_steps"),
    "batch_size": config["data"]["batch_size"],
    "gradient_accumulation_steps": grad_accum,
    "effective_micro_batch_x_accum": config["data"]["batch_size"] * grad_accum,
    "num_workers": config["data"]["num_workers"],
    "context_length": config["model"]["context_length"],
    "stride": config["model"].get("stride"),
    "target_vram_gb_per_gpu": config["runtime"].get("target_vram_gb"),
    "runtime": RUNTIME,
}
config_summary


## VRAM Check

Before training, verify that the config fits within your GPU's VRAM budget.
This cell loads the real `tokenizer_map.json`, instantiates the model, and estimates memory usage.

In [ ]:
# -- VRAM Preflight Check -----------------------------------------------------
from libs.data.training.tokenizer import SequenceTokenizer
from libs.core.pretrain.protein_lm.memory import (
    estimate_protein_pretrain_memory,
    _resolve_dtype_from_mixed_precision,
)
from libs.core.pretrain.protein_lm.support.backbone import (
    build_mdc_config_from_progen_config,
    build_progen_config,
)

# Load real tokenizer
tokenizer_map_path = config["paths"]["tokenizer_map_path"]
assert tokenizer_map_path.exists(), f"tokenizer_map.json not found: {tokenizer_map_path}"
tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
print(f"Tokenizer loaded: vocab_size = {tokenizer.vocab_size}")

# Build model config with mixed precision dtype
mixed_precision = config["runtime"]["mixed_precision"]
resolved_dtype = _resolve_dtype_from_mixed_precision(mixed_precision)
model_cfg = config["model"]
progen_config = build_progen_config(
    model_cfg["progen_model_size"],
    vocab_size=tokenizer.vocab_size,
    context_length=model_cfg["context_length"],
    dtype=resolved_dtype,
)
overrides = model_cfg["progen_config_overrides"]
if overrides:
    progen_config = {**progen_config, **overrides}
model_config = build_mdc_config_from_progen_config(progen_config, dtype=resolved_dtype)

# Estimate VRAM. For notebook DataParallel this is conservative because the
# actual micro-batch is split across GPUs by torch.nn.DataParallel.
import torch
estimate = estimate_protein_pretrain_memory(
    model_config=model_config,
    tokenizer=tokenizer,
    batch_size=config["data"]["batch_size"],
    context_length=model_cfg["context_length"],
    optimizer_type=config["optimizer"]["type"],
    dtype=resolved_dtype,
    mixed_precision=mixed_precision,
)

runtime_cfg = config.get("runtime", {})
configured_target_vram_gb = runtime_cfg.get("target_vram_gb")

if configured_target_vram_gb is not None:
    target_budget = float(configured_target_vram_gb)
    budget_source = f"{CONFIG_PATH.name}: runtime.target_vram_gb"
elif torch.cuda.is_available():
    detected_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    target_budget = min(detected_vram_gb * 0.85, detected_vram_gb - 2.0)
    budget_source = "detected CUDA VRAM with safety margin"
else:
    raise RuntimeError("Set runtime.target_vram_gb in the training YAML before running VRAM preflight without CUDA.")

visible_gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print()
print(f"{'='*60}")
print(f"VRAM ESTIMATE (resolved_vocab_size={tokenizer.vocab_size})")
print(f"{'='*60}")
print(f"  Source config:    {CONFIG_SOURCE_PATH}")
print(f"  Effective config: {CONFIG_PATH}")
print(f"  Multi-GPU mode:   {config['runtime']['multi_gpu_mode']}")
print(f"  Visible GPUs:     {visible_gpu_count}")
print(f"  Parameters:       {estimate['param_count']:,} ({estimate['param_memory_gb']:.3f} GB)")
print(f"  Gradients:        {estimate['gradient_memory_gb']:.3f} GB")
print(f"  Optimizer state:  {estimate['optimizer_state_gb']:.3f} GB")
print(f"  Activations:      {estimate['activation_memory_gb']:.3f} GB")
print(f"  Logits:           {estimate['logits_memory_gb']:.3f} GB")
print(f"  TOTAL:            {estimate['total_estimate_gb']:.3f} GB")
print(f"  Budget:           {target_budget:.3f} GB per GPU ({budget_source})")
print(f"  Margin:           {target_budget - estimate['total_estimate_gb']:+.3f} GB")
if torch.cuda.is_available():
    for idx in range(visible_gpu_count):
        total_vram_gb = torch.cuda.get_device_properties(idx).total_memory / (1024**3)
        print(f"  GPU {idx} VRAM:      {total_vram_gb:.3f} GB")

fits = estimate["total_estimate_gb"] <= target_budget
if fits:
    print()
    print(f"  OK: estimated to fit within configured budget ({target_budget:.1f}GB per GPU)")
else:
    print()
    print(f"  WARNING: exceeds configured budget ({target_budget:.1f}GB per GPU)")

if not torch.cuda.is_available():
    print("  No CUDA detected: this is only an estimate. Actual peak may be higher.")


In [ ]:
# -- Final config gate --------------------------------------------------------
print(f"Using config: {CONFIG_SOURCE_PATH}")
print(f"  effective_config={CONFIG_PATH}")
if CONFIG_NOTEBOOK_OVERRIDES:
    print(f"  notebook_overrides={CONFIG_NOTEBOOK_OVERRIDES}")
print(f"  batch_size={config['data']['batch_size']}, "
      f"context_length={config['model']['context_length']}, "
      f"grad_accum={config['training'].get('gradient_accumulation_steps', 1)}, "
      f"multi_gpu={config['runtime']['multi_gpu_mode']}")

preflight = config.get("runtime", {}).get("preflight_vram_check", False) if isinstance(config.get("runtime"), dict) else False
if not fits and preflight:
    raise RuntimeError(
        f"Config estimated to use {estimate['total_estimate_gb']:.2f}GB "
        f"which exceeds target budget of {target_budget:.2f}GB per GPU. "
        f"Increase runtime.target_vram_gb in the YAML or reduce batch_size/context_length."
    )
elif not fits:
    print(f"WARNING: Config may exceed target VRAM budget ({target_budget:.2f}GB per GPU).")


In [ ]:
# -- Train --------------------------------------------------------------------
trainer = ProteinPretrainTrainer(REPO_DIR, config_path=CONFIG_PATH)
result = trainer.train()
result


In [ ]:
# ── Generate sample ───────────────────────────────────────────────────────────
from libs.core.pretrain.protein_lm.core import generate_protein_text

if trainer.is_main_process and trainer.model is not None:
    sample = generate_protein_text(
        trainer.model,
        trainer.tokenizer,
        prompt="<|protein|>",
        device=trainer.runtime.device,
        max_new_tokens=80,
        context_length=int(trainer.model_config.context_length),
    )
    print(sample)
else:
    print("Sample generation is emitted on rank 0 only.")

In [ ]:
# ── Upload model to HuggingFace ───────────────────────────────────────────────
from huggingface_hub import HfApi, login

HF_REPO_ID = "admincybers2/MDNAC-protein-pretrain"  # ← change repo name as needed

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    print("HF_TOKEN is not set; skipping HuggingFace upload.")
else:
    login(token=hf_token)
    api = HfApi()

    checkpoint_path = result.checkpoint_path
    best_path = result.best_checkpoint_path

    files_to_upload = []
    if checkpoint_path and checkpoint_path.exists():
        files_to_upload.append((str(checkpoint_path), checkpoint_path.name))
    if best_path and best_path.exists():
        files_to_upload.append((str(best_path), best_path.name))

    tokenizer_map = config["paths"]["tokenizer_map_path"]
    if Path(tokenizer_map).exists():
        files_to_upload.append((str(tokenizer_map), "tokenizer_map.json"))

    resume_state = config["paths"]["resume_state_path"]
    if Path(resume_state).exists():
        files_to_upload.append((str(resume_state), "resume_state.json"))

    for local_path, repo_filename in files_to_upload:
        print(f"Uploading {repo_filename} ...")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_filename,
            repo_id=HF_REPO_ID,
            repo_type="model",
        )

    print(f"✅ Uploaded {len(files_to_upload)} file(s) to https://huggingface.co/{HF_REPO_ID}")